# Module 4 — RAG Evaluation with RAGAS

**Duration:** ~90 minutes  
**Goal:** Measure RAG quality systematically, interpret the metrics, and use them to diagnose and fix pipeline weaknesses.

---

## Why evaluation matters

Without measurement, RAG improvement is guesswork. You might improve faithfulness while  
degrading relevancy, and you'd never know.

**The problem:** RAG evaluation requires assessing *three* things simultaneously:
- Is the **retrieval** finding the right passages?
- Is the **answer** faithful to what was retrieved?
- Is the **answer** actually helpful for the question?

---

## RAGAS Metrics

| Metric | What it measures | Needs ground truth? |
|--------|-----------------|--------------------|
| **Faithfulness** | Is the answer supported by the retrieved context? | No |
| **Answer Relevancy** | Does the answer address the question? | No |
| **Context Precision** | Are the retrieved chunks relevant to the question? | Yes |
| **Context Recall** | Does the retrieved context cover the ground truth? | Yes |

### Interpreting scores

```
Faithfulness = 0.9    → Answer stays close to context (good for fact-heavy tasks)
Faithfulness = 0.4    → Answer invents facts not in context (hallucination!)

Context Recall = 0.9  → Retrieved chunks contain most of the ground truth
Context Recall = 0.3  → Wrong chunks retrieved — retrieval is failing

Answer Relevancy = 0.9 → Answer directly addresses the question
Answer Relevancy = 0.5 → Answer is tangential or too long
```

### The Four Failure Modes

```
                    Context Recall
                  High        Low
                ┌───────────┬────────────┐
  Faithfulness  │           │            │
  High          │  GOOD ✓   │  Retrieval │
                │           │  failure   │
                ├───────────┼────────────┤
  Faithfulness  │ Hallucin- │  Double    │
  Low           │ ation     │  failure   │
                └───────────┴────────────┘
```

In [ ]:
import json
import os
import warnings
warnings.filterwarnings("ignore")

from openai import OpenAI
import chromadb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

load_dotenv("../.env")

# Re-use the RAGPipeline from Module 3 (copy here for self-containedness)
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
chroma_client = chromadb.PersistentClient(path="../chroma_db")
collection = chroma_client.get_or_create_collection(
    name="workshop_rag", metadata={"hnsw:space": "cosine"}
)
llm_client = OpenAI(
    base_url=os.getenv("NETLIGHT_BASE_URL", "https://llm.netlight.ai/v1"),
    api_key=os.getenv("NETLIGHT_API_KEY"),
)
FAST_MODEL = os.getenv("NETLIGHT_MODEL_FAST", "claude-haiku-4-5")

print(f"Collection: {collection.count()} docs")
if collection.count() == 0:
    print("Collection empty — rebuilding from corpus (fallback for Day 2)...")
    import sys; sys.path.insert(0, "..")
    from src.chunker import build_chunk_records
    from src.vector_store import index_chunks
    with open("../data/raw/corpus.json") as _f:
        _corpus = json.load(_f)
    _chunks = build_chunk_records(_corpus, chunk_size=800, chunk_overlap=100)
    index_chunks(collection, _chunks, embed_model)
    print(f"Rebuilt: {collection.count()} docs indexed")


In [ ]:
# Paste the final RAGPipeline from Module 3 here
SYSTEM_PROMPT = """\
You are a knowledgeable insurance and technology assistant.
Answer ONLY using information from the numbered context chunks provided.
Cite sources using the chunk number, e.g. "According to [1]..."
If the context is insufficient, say so explicitly.
Keep answers concise. Never fabricate facts."""


def build_prompt(question: str, chunks: list[dict]) -> str:
    lines = []
    for i, chunk in enumerate(chunks, 1):
        lines.append(f"[{i}] Source: {chunk['source']} (relevance: {chunk.get('similarity', 0):.2f})")
        lines.append(chunk["text"])
        lines.append("")
    return f"Retrieved Context:\n{''.join(lines)}\nQuestion: {question}\n\nAnswer:"


class RAGPipeline:
    def __init__(self, collection, embed_model, llm_client, model=FAST_MODEL,
                 top_k=5, system_prompt=SYSTEM_PROMPT, min_similarity=0.0):
        self.collection = collection
        self.embed_model = embed_model
        self.llm_client = llm_client
        self.model = model
        self.top_k = top_k
        self.system_prompt = system_prompt
        self.min_similarity = min_similarity

    def retrieve(self, question: str) -> list[dict]:
        q_emb = self.embed_model.encode(question).tolist()
        results = self.collection.query(
            query_embeddings=[q_emb], n_results=self.top_k,
            include=["documents", "metadatas", "distances"]
        )
        chunks = [
            {"text": t, "source": m["source"], "similarity": 1 - d}
            for t, m, d in zip(results["documents"][0], results["metadatas"][0], results["distances"][0])
        ]
        return [c for c in chunks if c["similarity"] >= self.min_similarity]

    def ask(self, question: str) -> dict:
        chunks = self.retrieve(question)
        prompt = build_prompt(question, chunks)
        response = self.llm_client.chat.completions.create(
            model=self.model,
            max_tokens=1024,
            messages=[
                {"role": "system", "content": self.system_prompt},
                {"role": "user", "content": prompt},
            ],
        )
        return {
            "question": question,
            "answer": response.choices[0].message.content,
            "contexts": [c["text"] for c in chunks],
            "sources": [c["source"] for c in chunks],
        }


rag = RAGPipeline(collection=collection, embed_model=embed_model, llm_client=llm_client)
print("RAGPipeline ready")

---
## Part 1 — Build the Evaluation Dataset

### Exercise 4.1 — Generate answers for all test questions

> **Task:** Run `rag.ask()` on all questions in `data/sample_qa.json` and collect the results  
> in the format RAGAS expects.
>
> RAGAS needs for each example:
> - `user_input` — the question asked
> - `response` — the answer your RAG produced
> - `retrieved_contexts` — the list of text chunks that were retrieved
> - `reference` — the ground truth answer
>
> **Hint:** Save the results to avoid re-running the LLM every time you tweak evaluation code.

In [ ]:
from pathlib import Path
from tqdm import tqdm

# Load ground truth
with open("../data/sample_qa.json") as f:
    qa_data = json.load(f)

CACHE_PATH = Path("../data/rag_results.json")

# TODO: check if cached results exist and load them
# If not, run the pipeline on all questions and save

if CACHE_PATH.exists():
    # TODO: load from cache
    rag_results = ...
    print(f"Loaded {len(rag_results)} cached results")
else:
    rag_results = []
    print("Running RAG on all questions (this may take a minute)...")
    for qa in tqdm(qa_data["questions"]):
        # TODO: call rag.ask() and append result with ground_truth
        pass
    # TODO: save to CACHE_PATH

print(f"\nFirst result keys: {list(rag_results[0].keys()) if rag_results else 'empty'}")

### Exercise 4.2 — Build the RAGAS EvaluationDataset

> **Task:** Convert `rag_results` into a RAGAS `EvaluationDataset`.
>
> **Hints:**
> ```python
> from ragas import EvaluationDataset
> # EvaluationDataset expects a list of dicts with keys:
> # user_input, response, retrieved_contexts, reference
> dataset = EvaluationDataset.from_list([...])
> ```

In [ ]:
from ragas import EvaluationDataset

# TODO: build the dataset
eval_data = [
    # {
    #     "user_input": result["question"],
    #     "response": result["answer"],
    #     "retrieved_contexts": result["contexts"],
    #     "reference": result["ground_truth"],
    # }
    # for result in rag_results
]

# dataset = EvaluationDataset.from_list(eval_data)
# print(f"EvaluationDataset: {len(dataset)} examples")

---
## Part 2 — Run RAGAS Evaluation

### Exercise 4.3 — Evaluate with all four metrics

> **Task:** Run RAGAS evaluation with the four metrics.
> RAGAS uses an LLM internally to compute some metrics — we'll use Claude.
>
> **Hints:**
> ```python
> from ragas import evaluate
> from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
> from ragas.llms import LangchainLLMWrapper  # RAGAS uses langchain internally
> from langchain_openai import ChatOpenAI
> ```
>
> **Note:** RAGAS uses an LLM as a judge. This takes a few minutes and costs tokens.
> Use `claude-haiku-4-5` to keep costs low.

In [ ]:
# Exercise 4.3 — Run evaluation
from src.evaluator import run_evaluation

# TODO: call run_evaluation() with your rag_results and llm_client
# df = run_evaluation(rag_results, llm_client=llm_client, judge_model=FAST_MODEL)

# Print mean scores across all questions:
# print(df[["faithfulness", "answer_relevancy", "context_precision", "context_recall"]].mean().round(3))


---
## Part 3 — Analyse Results

### Exercise 4.4 — Visualise metrics and find weak spots

> **Task:** Convert results to a DataFrame and:
> 1. Show a bar chart of mean scores for each metric
> 2. Find the 3 questions with the lowest faithfulness scores
> 3. Find the 3 questions with the lowest context recall
>
> **Hint:** `results.to_pandas()` returns a DataFrame with per-question scores.

In [ ]:
# TODO: convert results to DataFrame
# df = results.to_pandas()
# print(df.head())

# TODO: plot mean metric scores
# metrics = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
# means = df[metrics].mean()
# ...

# Placeholder visualisation (replace with real data)
import numpy as np

# Simulate what results look like so you can build visualisation code now
np.random.seed(42)
n = 15
simulated = pd.DataFrame({
    "question": [f"Q{i+1}" for i in range(n)],
    "faithfulness": np.clip(np.random.normal(0.75, 0.15, n), 0, 1),
    "answer_relevancy": np.clip(np.random.normal(0.80, 0.12, n), 0, 1),
    "context_precision": np.clip(np.random.normal(0.65, 0.20, n), 0, 1),
    "context_recall": np.clip(np.random.normal(0.70, 0.18, n), 0, 1),
})

metrics = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
means = simulated[metrics].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of mean scores
colors = ["#e74c3c" if v < 0.6 else "#f39c12" if v < 0.75 else "#27ae60" for v in means]
axes[0].bar(means.index, means.values, color=colors, edgecolor="white", linewidth=1.5)
axes[0].set_ylim(0, 1)
axes[0].axhline(0.6, color="red", linestyle="--", alpha=0.5, label="Caution threshold")
axes[0].axhline(0.75, color="orange", linestyle="--", alpha=0.5, label="Good threshold")
axes[0].set_title("Mean RAGAS Scores (simulated)")
axes[0].set_ylabel("Score (0–1)")
axes[0].legend(fontsize=8)
for i, (m, v) in enumerate(zip(means.index, means.values)):
    axes[0].text(i, v + 0.01, f"{v:.2f}", ha="center", fontsize=9)

# Heatmap of per-question scores
heat_data = simulated.set_index("question")[metrics]
sns.heatmap(heat_data, ax=axes[1], cmap="RdYlGn", vmin=0, vmax=1,
            annot=True, fmt=".2f", linewidths=0.5)
axes[1].set_title("Per-Question RAGAS Scores (simulated)")

plt.tight_layout()
plt.show()

print("\nBottom 3 by faithfulness:")
print(simulated.nsmallest(3, "faithfulness")[["question", "faithfulness"]].to_string(index=False))

---
## Part 4 — Diagnose and Fix

### Exercise 4.5 — Improve a specific failure case

> **Task:** Take one of the low-scoring questions and investigate why it scored poorly.  
> Then modify the pipeline configuration to improve its score.
>
> **Diagnostic checklist:**
> - Print the retrieved chunks for the failing question — are they relevant?
> - Print the generated answer — does it introduce facts not in the context?
> - Try increasing `top_k` — does it help context recall?
> - Try a stricter `min_similarity` threshold — does it improve precision?
> - Rewrite the system prompt to be more restrictive — does it improve faithfulness?

In [ ]:
# Investigate a specific failing question
# Replace with an actual low-scoring question from your results

failing_question = "Is laser eye surgery covered under DKV hospitalisation insurance?"

# Step 1: See what was retrieved
result = rag.ask(failing_question)
print(f"Question: {failing_question}\n")
print(f"Answer:\n{result['answer']}\n")
print(f"Retrieved {len(result['contexts'])} chunks from: {set(result['sources'])}")

print("\n--- Retrieved chunks ---")
for i, (text, source) in enumerate(zip(result["contexts"], result["sources"]), 1):
    print(f"[{i}] {source}: {text[:200]}...\n")

In [ ]:
# Step 2: Try a modified pipeline and compare

# TODO: create a rag_improved with different parameters
# Options:
#   - top_k=7 instead of 5
#   - min_similarity=0.35 to filter noise
#   - SMART_MODEL for better reasoning
#   - Rewritten system prompt

rag_improved = RAGPipeline(
    collection=collection,
    embed_model=embed_model,
    llm_client=llm_client,
    model=FAST_MODEL,
    top_k=5,        # TODO: try changing this
    system_prompt=SYSTEM_PROMPT,  # TODO: try a different prompt
)

result_improved = rag_improved.ask(failing_question)
print(f"Improved answer:\n{result_improved['answer']}")

### Exercise 4.6 — Run a mini A/B evaluation

> **Task:** Run both the baseline and improved pipeline on all 15 questions.  
> Compare their RAGAS scores in a side-by-side chart.
>
> **Goal:** Show that improving one configuration dimension has a measurable effect on metrics — and may have trade-offs.

In [ ]:
# TODO: Run both pipelines on all questions
# Compare RAGAS scores

# Simulated A/B comparison (replace with real scores)
ab_data = pd.DataFrame({
    "metric": metrics * 2,
    "score": [
        0.75, 0.80, 0.65, 0.70,   # baseline
        0.82, 0.78, 0.72, 0.76,   # improved
    ],
    "pipeline": ["Baseline"] * 4 + ["Improved"] * 4,
})

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(metrics))
width = 0.35
baseline_scores = ab_data[ab_data["pipeline"]=="Baseline"]["score"].values
improved_scores = ab_data[ab_data["pipeline"]=="Improved"]["score"].values

ax.bar([i - width/2 for i in x], baseline_scores, width, label="Baseline", color="#3498db", alpha=0.8)
ax.bar([i + width/2 for i in x], improved_scores, width, label="Improved", color="#2ecc71", alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("Baseline vs Improved Pipeline (simulated)")
ax.legend()
plt.tight_layout()
plt.show()

print("\nKey takeaway: improving one metric can hurt another.")
print("This is why systematic evaluation is essential — not just eyeballing answers.")

---
## Part 5 — Alternative Evaluation with promptfoo

**promptfoo** is a CLI tool for testing and evaluating LLM applications.  
It differs from our direct evaluator in two key ways:

| | Direct LLM evaluator | promptfoo |
|---|---|---|
| Output | Numeric scores (0–1) | Pass / Fail + score |
| Config | Python code | YAML file (version-controlled) |
| Report | DataFrame / chart | Interactive HTML UI |
| CI/CD | Custom script | Built-in `--output json` |

We use the **pre-computed `rag_results`** from Part 1 so the pipeline doesn't re-run.  
An `echo` provider passes each answer straight to the LLM judge.


### Exercise 4.7 — Generate the promptfoo config

> **Task:** Implement `write_promptfoo_config()` in `src/evaluator.py`, then call it here.
>
> The function should produce a `promptfooconfig.yaml` with one test per question,  
> using an `llm-rubric` assertion that checks faithfulness and relevance.
>
> **Hints:**
> ```python
> from src.evaluator import write_promptfoo_config
> config_path = write_promptfoo_config(
>     rag_results,
>     output_path="../promptfooconfig.yaml",
>     judge_model=FAST_MODEL,
> )
> ```


In [ ]:
from src.evaluator import write_promptfoo_config

# TODO: call write_promptfoo_config() and print the config path
config_path = ...
print(f"Config written to: {config_path}")

# Inspect the generated YAML:
# with open(config_path) as f:
#     print(f.read()[:800])

import subprocess, os
from dotenv import dotenv_values

# TODO: run promptfoo eval — pass .env values so the LLM judge can reach the endpoint
_dotenv = dotenv_values("../.env")
_env = {
    **os.environ,
    **_dotenv,
    "OPENAI_API_KEY": _dotenv.get("NETLIGHT_API_KEY", ""),
    "OPENAI_BASE_URL": _dotenv.get("NETLIGHT_BASE_URL", "").rstrip("/"),
}

proc = subprocess.run(
    ["npx", "--yes", "promptfoo@0.82.0", "eval",
     "--config", "../promptfooconfig.yaml",
     "--output", "../data/promptfoo_results.json"],
    capture_output=True, text=True, env=_env,
)
print(proc.stdout[-2000:] if len(proc.stdout) > 2000 else proc.stdout)
if proc.returncode != 0:
    print("stderr:", proc.stderr[-1000:])

In [ ]:
import subprocess, os
from dotenv import dotenv_values

# TODO: run promptfoo eval — pass .env values so the LLM judge can reach the endpoint
_dotenv = dotenv_values("../.env")
_env = {
    **os.environ,
    **_dotenv,
    "OPENAI_API_KEY": _dotenv.get("NETLIGHT_API_KEY", ""),
    "OPENAI_BASE_URL": _dotenv.get("NETLIGHT_BASE_URL", "").rstrip("/"),
}

proc = subprocess.run(
    ["npx", "--yes", "promptfoo@0.82.0", "eval",
     "--config", "../promptfooconfig.yaml",
     "--output", "../data/promptfoo_results.json"],
    capture_output=True, text=True, env=_env,
)
print(proc.stdout[-2000:] if len(proc.stdout) > 2000 else proc.stdout)
if proc.returncode != 0:
    print("stderr:", proc.stderr[-1000:])

In [ ]:
# Parse promptfoo results into a DataFrame
from pathlib import Path

pf_path = Path("../data/promptfoo_results.json")
if not pf_path.exists():
    print("promptfoo_results.json not found — run the eval cell above first.")
else:
    with open(pf_path) as f:
        pf_data = json.load(f)

    rows = []
    for r in pf_data.get("results", {}).get("results", []):
        rows.append({
            "question": r.get("vars", {}).get("question", "")[:60],
            "pass": r.get("gradingResult", {}).get("pass", False),
            "score": r.get("gradingResult", {}).get("score", 0.0),
            "reason": r.get("gradingResult", {}).get("reason", ""),
        })

    df_pf = pd.DataFrame(rows)
    print(f"Pass rate: {df_pf['pass'].mean():.1%}")
    display(df_pf[["question", "pass", "score"]])

> **View the promptfoo report in your terminal:**
> ```bash
> npx promptfoo@0.82.0 view
> ```
> Then open http://localhost:15500 in your browser.


---
## Reflection Questions

1. **RAGAS uses an LLM as a judge.** Is that reliable? What are the failure modes?

2. **Ground truth quality:** Our `sample_qa.json` ground truths were hand-written.  
   In a real project, where would ground truth come from?

3. **Metric weighting:** For a customer-facing insurance chatbot, which metric matters most?  
   Would your answer change for an internal analyst tool?

4. **Evaluation cost:** Running RAGAS on 15 questions costs ~$0.05.  
   How would you structure evaluation for a production system with thousands of queries?

5. **Online vs offline evaluation:** RAGAS is offline (batch). What would *online* evaluation look like?
   (Hint: user feedback, thumbs up/down, escalation rates)

---

## What we built

- [x] RAGAS evaluation pipeline with 4 metrics
- [x] Per-question score analysis
- [x] Failure mode diagnosis
- [x] A/B comparison framework

## Next: Module 5 — Advanced RAG

Equipped with evaluation, we can now measure the impact of advanced techniques.  
Open `06_experiments.ipynb` →